# QC the Tractor-Mix GRM

Summarizes pairwise kinship from MakeGRM outputs and (optionally) checks
close pairs against `pedigree_family_id` in `covariates.source_rebuilt.csv.gz`.

**Kinship scale:** PLINK2 `--make-rel` (GCTA-like). Rough expectations:

| Kinship | Relationship |
|---------|--------------|
| ~0.5 | full sib / parent–offspring |
| ~0.25 | half-sib / grandparent / avuncular |
| ~0.125 | first cousin |
| ≥0.707 | duplicate / MZ twin |

For the **current** successful run (before richer MakeGRM outputs), point
`GRM_SPARSE_RDS` at the workflow `grm_sparse.rds` and leave the TSV paths blank.
Future runs also emit `grm_relationship_bands.tsv`, `grm_kinship_histogram.tsv`,
and `grm_close_pairs.tsv` directly from MakeGRM.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Bootstrap scripts/ from $WORKSPACE_BUCKET/scripts/ when not on the VM.
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook

SCRIPTS = init_notebook("workspace_paths.py", "plot_grm_qc.R")
from workspace_paths import data_root

import shutil
import subprocess

ROOT = data_root()
ws = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")

GRM_SPARSE_RDS = os.environ.get("TRACTOR_GRM_SPARSE_RDS", "")
GRM_BANDS = os.environ.get("TRACTOR_GRM_BANDS", "")
GRM_HIST = os.environ.get("TRACTOR_GRM_HIST", "")
GRM_CLOSE = os.environ.get("TRACTOR_GRM_CLOSE", "")
COV_GCS = os.environ.get(
    "TRACTOR_COVARIATES_GCS",
    f"{ws}/covariates/covariates.source_rebuilt.csv.gz" if ws else "",
)

OUT = Path(os.environ.get("GRM_QC_OUT", "grm_qc"))
OUT.mkdir(parents=True, exist_ok=True)
LOCAL = Path(os.environ.get("GRM_QC_WORK", "grm_qc_inputs"))
LOCAL.mkdir(parents=True, exist_ok=True)


def pull(uri: str, name: str) -> Path | None:
    if not uri:
        return None
    dest = LOCAL / name
    if dest.exists():
        return dest
    if uri.startswith("gs://"):
        subprocess.check_call(["gsutil", "cp", uri, str(dest)])
    else:
        shutil.copy2(uri, dest)
    return dest


assert GRM_SPARSE_RDS, "Set TRACTOR_GRM_SPARSE_RDS to the MakeGRM grm_sparse.rds URI"
rds = pull(GRM_SPARSE_RDS, "grm_sparse.rds")
bands = pull(GRM_BANDS, "grm_relationship_bands.tsv")
hist = pull(GRM_HIST, "grm_kinship_histogram.tsv")
close = pull(GRM_CLOSE, "grm_close_pairs.tsv")
cov = pull(COV_GCS, "covariates.source_rebuilt.csv.gz") if COV_GCS else None
print("inputs:", {"rds": rds, "bands": bands, "hist": hist, "close": close, "cov": cov})


In [ ]:
cmd = [
    "Rscript", str(SCRIPTS / "plot_grm_qc.R"),
    "--sparse-rds", str(rds),
    "--out-dir", str(OUT),
]
if bands:
    cmd += ["--bands", str(bands)]
if hist:
    cmd += ["--histogram", str(hist)]
if close:
    cmd += ["--close-pairs", str(close)]
if cov:
    cmd += ["--covariates", str(cov)]

print(" ".join(cmd))
import subprocess
from IPython.display import Image, display
import pandas as pd

# subprocess: IPython system() often returns None/nonzero even when R succeeds
proc = subprocess.run(cmd, check=False)
if proc.returncode != 0:
    raise RuntimeError(f"plot_grm_qc.R failed with exit code {proc.returncode}")

display(pd.read_csv(OUT / "grm_relationship_bands.tsv", sep="\t"))
ped = OUT / "grm_pedigree_concordance.tsv"
if ped.exists():
    display(pd.read_csv(ped, sep="\t"))
display(Image(filename=str(OUT / "grm_kinship_histogram.png")))
display(Image(filename=str(OUT / "grm_relationship_bands.png")))